# PII Masking with NLP

### View Data

In [ ]:
import pandas as pd

df_full = pd.read_csv('PII43k.csv', usecols=['Template', 'Filled Template'])


def get_test_set(df, test_pct, random_state=42):
    # Sample the remaining fraction (i.e. non-test) and return the test set
    train = df.sample(frac=1 - test_pct, random_state=random_state)
    return df.drop(train.index)

# Create 10 datasets with test percentages from 10% to 100%
test_pcts = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 1.00]

# Build list of datasets
df_list = [get_test_set(df_full, pct, random_state=42) for pct in test_pcts]

class dataset:
    def __init__(self, name=None, dataset=None, text_accuracy=0.0, label_accuracy=0.0, nlp=None):
        self.name = name if name is not None else ""
        self.dataset = dataset if dataset is not None else pd.DataFrame()
        self.failed_labels = []
        self.text_accuracy = text_accuracy  
        self.label_accuracy = label_accuracy
        self.nlp = nlp

    def __repr__(self):
        return (f"dataset(name={self.name}, " # TODO: Rename to size and add a new name field with version name.
                f"dataset={self.dataset}, "
                f"failed_labels={self.failed_labels}, "
                f"text_accuracy={self.text_accuracy:.2%}, "
                f"label_accuracy={self.label_accuracy:.2%}, "
                f"nlp={self.nlp})")

datasets = {}
for name, ds in zip(test_pcts, df_list):
    # Create a dataset instance with default (empty) init values.
    result = dataset(name, ds)
    datasets[name] = result

print(datasets)

In [ ]:
df_full.head()

In [ ]:
import re

# Use df_ground_truth as the dataset for ground truth
# Regex to extract values inside square brackets from each row in df_ground_truth
unique_matches = set()
for text in df_full["Template"].dropna():
    matches = re.findall(r'\[([^]]+)\]', text)
    unique_matches.update(matches)

print(unique_matches)
print(len(unique_matches))

# Clean the unique_matches set by removing the trailing underscore and everything
cleaned_matches = {re.sub(r'_[^_]+$', '', token) for token in unique_matches}


print(cleaned_matches)
print(len(cleaned_matches))


In [ ]:
# Count occurrences of each cleaned token in df_full
counts = {}
for token in cleaned_matches:
    pattern = r'\[' + token + r'_\d+\]'
    counts[token] = int(df_full["Template"].dropna().str.count(pattern).sum())

# Convert to a DataFrame and sort by count for a nicer display
counts_df = pd.DataFrame(list(counts.items()), columns=['Token', 'Count']).sort_values(by='Count', ascending=False)
counts_df

### NLP Implementation 

In [5]:
from presidio_analyzer import AnalyzerEngine

analyzer = AnalyzerEngine()

In [ ]:
# test Call analyzer to get results
results = analyzer.analyze(text="My phone number is 212-555-5555",
                           entities=["PHONE_NUMBER"],
                           language='en')
print(results)

In [7]:
def clean_text(text):
    """Clean and normalize text."""
    return str(text).strip()

def extract_entities(template, filled):
    """
    Extract entities by aligning a template (with placeholders) to the filled text.
    
    Assumes that the filled text is identical to the template except that each placeholder 
    (e.g. "[NAME_1]") has been replaced by the actual value.
    
    Returns:
        A list of tuples (start_char, end_char, label) for entities in the filled text.
    """
    entities = []
    i = 0  # pointer for template
    j = 0  # pointer for filled text

    while i < len(template) and j < len(filled):
        if template[i] == '[':
            # Found a placeholder in the template.
            closing = template.find(']', i)
            if closing == -1:
                break  # malformed template (no matching ])
            # Extract the raw placeholder, e.g. "[NAME_1]"
            # Remove the brackets and any trailing digits to get the label.
            label_raw = template[i+1:closing]    # e.g. "NAME_1"
            label = re.sub(r'_\d+', '', label_raw)  # e.g. becomes "NAME"
            
            # Determine the literal text that follows the placeholder in the template.
            next_i = closing + 1
            next_bracket = template.find('[', next_i)
            literal = template[next_i:] if next_bracket == -1 else template[next_i:next_bracket]
            
            # In the filled text, the actual entity value replaces the placeholder.
            # We assume that the literal following the placeholder appears unchanged.
            if literal:
                literal_index = filled.find(literal, j)
            else:
                literal_index = len(filled)
            
            if literal_index == -1:
                # If we cannot find the literal, assume the entity is the rest of the filled text.
                entity_start = j
                entity_end = len(filled)
                j = len(filled)
            else:
                entity_start = j
                entity_end = literal_index
                j = literal_index  # advance pointer j to the beginning of the literal
            
            entities.append((entity_start, entity_end, label))
            # Advance pointer i past the entire placeholder.
            i = closing + 1
        else:
            # For non-placeholder characters, assume they match between template and filled.
            if template[i] == filled[j]:
                i += 1
                j += 1
            else:
                # If there is a mismatch (e.g. extra whitespace), increment j.
                j += 1
    return entities


In [8]:
import pandas as pd
import random
import spacy
from spacy.training.example import Example
from spacy.util import minibatch


# TODO: Run later: i want to collect F1 score, accuracy, false postetives, false negatives etc. everything

def NLP_training(df):
    # Build initial training data by aligning each template with its filled version.
    raw_train_data = []
    for _, row in df.iterrows():
        template = clean_text(row['Template'])
        filled = clean_text(row['Filled Template'])
        entities = extract_entities(template, filled)
        if entities:
            # Each training example is a tuple: (text, {"entities": [(start, end, label), ...]})
            raw_train_data.append((filled, {"entities": entities}))

    # ------------------------------
    # Step 0.5: Re-align Entity Offsets to Token Boundaries
    # ------------------------------
    tokenizer_nlp = spacy.blank("en")
    aligned_train_data = []
    for text, annotation in raw_train_data:
        doc = tokenizer_nlp(text)
        new_entities = []
        for start, end, label in annotation["entities"]:
            # Use "expand" mode to adjust the span to token boundaries.
            span = doc.char_span(start, end, alignment_mode="expand")
            if span is not None:
                new_entities.append((span.start_char, span.end_char, label))
            else:
                # If alignment fails, you might choose to log or skip the entity.
                print(f"WARNING: Could not align entity '{text[start:end]}' in text: {text}")
        if new_entities:
            aligned_train_data.append((text, {"entities": new_entities}))
    # Use the aligned data for training.
    TRAIN_DATA = aligned_train_data

    # ------------------------------
    # Step 1: Split Data
    # ------------------------------
    # Here we use an 80/20 train/validation split.
    train_size = int(0.8 * len(TRAIN_DATA))
    train_data = TRAIN_DATA[:train_size]
    valid_data = TRAIN_DATA[train_size:]

    # ------------------------------
    # Step 2: Create and Configure the Model
    # ------------------------------
    nlp = spacy.blank("en")

    # Add a Named Entity Recognizer (NER) pipeline component if not already present.
    if "ner" not in nlp.pipe_names:
        ner = nlp.add_pipe("ner", last=True)
    else:
        ner = nlp.get_pipe("ner")

    # Add each entity label from the training data to the NER component.
    for _, annotations in train_data:
        for start, end, label in annotations["entities"]:
            ner.add_label(label)

    # ------------------------------
    # Step 3: Train the Model Using Batches with Dropout
    # ------------------------------
    optimizer = nlp.begin_training()
    n_iter = 20  # Number of epochs
    batch_size = 16

    for itn in range(n_iter):
        random.shuffle(train_data)
        batches = minibatch(train_data, size=batch_size)
        losses = {}
        for batch in batches:
            examples = []
            for text, annotations in batch:
                doc = nlp.make_doc(text)
                examples.append(Example.from_dict(doc, annotations))
            nlp.update(examples, sgd=optimizer, drop=0.3, losses=losses)
        print(f"Iteration {itn + 1}/{n_iter} - Losses: {losses}")

    # ------------------------------
    # Step 4: Define Improved Masking Function
    # ------------------------------
    def mask_pii(text, model):
        """
        Mask detected entities in the text with their label names.
        
        Entities are replaced starting from the end of the text (to avoid offset issues).
        """
        doc = model(text)
        spans = [(ent.start_char, ent.end_char, ent.label_) for ent in doc.ents]
        # Sort spans in reverse order of start index.
        spans = sorted(spans, key=lambda x: x[0], reverse=True)
        masked_text = text
        for start, end, label in spans:
            masked_text = masked_text[:start] + label + masked_text[end:]
        return masked_text

    # ------------------------------
    # Step 5: Evaluate and Print Combined Results
    # ------------------------------
    correct_texts = 0
    correct_labels = 0
    total_texts = 0
    total_labels = 0
    failed_labels = []

    print("\n=== Evaluation on Validation Data ===\n")
    for text, annotations in valid_data:
        masked_text = mask_pii(text, nlp)
        
        print("Original Text:")
        print(text)
        print("Masked Text:")
        print(masked_text)
        print("-" * 40)
        
        text_correct = True
        # Evaluate masking on each individual entity.
        for start, end, label in annotations["entities"]:
            total_labels += 1
            if label in masked_text:
                correct_labels += 1
            else:
                text_correct = False
                failed_labels.append((label, text[start:end]))
        
        if text_correct:
            correct_texts += 1
        total_texts += 1

    if failed_labels:
        print("\nFAILED MASKINGS:")
        for label, value in failed_labels:
            print(f"Label: {label}, Expected Value: {value}")
    else:
        print("\nAll entities were successfully masked in every text!")

    text_accuracy = correct_texts / total_texts if total_texts > 0 else 0
    label_accuracy = correct_labels / total_labels if total_labels > 0 else 0

    print(f"\nText Accuracy (all entities in a text masked correctly): {text_accuracy:.2%}")
    print(f"Label Accuracy (individual entity masking): {label_accuracy:.2%}")
    return text_accuracy, label_accuracy, failed_labels, nlp


In [ ]:
import pickle

for key, ds_obj in datasets.items():
    print(f"Processing dataset with test percentage: {ds_obj.name}")
    text_acc, label_acc, failed_labels, nlp_model = NLP_training(ds_obj.dataset)
    ds_obj.text_accuracy = text_acc
    ds_obj.label_accuracy = label_acc
    ds_obj.nlp = nlp_model
    print(ds_obj)

    with open(f"dataset_{ds_obj.name}.pkl", "wb") as file:
        pickle.dump(ds_obj, file)
    print(f"Saved ds_obj to dataset_{ds_obj.name}.pkl")

In [ ]:
import matplotlib.pyplot as plt

# Extract sorted list of dataset keys (test percentages)
sorted_keys = sorted(datasets.keys())

# Get accuracy values (convert to percentage)
text_accs = [datasets[k].text_accuracy * 100 for k in sorted_keys]
label_accs = [datasets[k].label_accuracy * 100 for k in sorted_keys]

# Create the plot
plt.figure(figsize=(8, 5))
plt.plot(sorted_keys, text_accs, marker='o', label='Text Accuracy')
plt.plot(sorted_keys, label_accs, marker='o', label='Label Accuracy')
plt.xlabel('Test Percentage')
plt.ylabel('Accuracy (%)')
plt.title('Accuracy Comparison Across Datasets')
plt.legend()
plt.grid(True)
plt.show()